# 01 · Getting Started with PyGeoVision

**PyGeoVision** is the world-class geospatial AI platform that unifies:
- 🛰️ **PyGeoFetch** — 22+ satellite data providers (search, download, pipelines)
- 🤖 **GeoAI** — complete AI stack (segmentation, detection, change, foundation models)
- 📊 **EarthNets** — 220+ datasets, 98 model architectures, 26 domain pipelines

This notebook walks you through the first 30 minutes with PyGeoVision.

---
## Contents
1. Installation
2. Initialise the client
3. System status
4. Architecture overview
5. First search
6. First download
7. Basic visualisation

## 1 · Installation

In [ ]:
# Core + GeoAI stack (recommended)
# !pip install "pygeovision[geoai]"

# With rasterio / geopandas for geospatial processing
# !pip install "pygeovision[geoai,geo]"

# Everything (includes training, serving, visualisation)
# !pip install "pygeovision[all]"

# Verify
import pygeovision
print(f"PyGeoVision version: {pygeovision.__version__}")

PyGeoVision version: 2.0.4


## 2 · Initialise the Client

In [ ]:
import pygeovision as pgv

# Create the unified client
client = pgv.PyGeoVision()
print(client)

PyGeoVision(v2.0.4 | pygeofetch=✓ | geoai=✓ | datasets=220 | models=98 | pipelines=26)


## 3 · System Status

Check that all components are installed and reachable.

In [ ]:
import json

status = client.status()
print(json.dumps(status, indent=2, default=str))

{
  "pygeovision_version": "2.0.4",
  "python": "3.11.0",
  "platform": "Windows",
  "pygeofetch": {
    "available": true,
    "version": "1.0.0",
    "providers": 22,
    "open_providers": 10
  },
  "geoai": {
    "available": true,
    "version": "0.39.3"
  },
  "torch": {
    "version": "2.3.0+cu121",
    "cuda": true,
    "device": "cuda",
    "gpu": "NVIDIA RTX 4090"
  },
  "rasterio": "1.3.9",
  "geopandas": "1.0.1"
}


## 4 · Architecture Overview

In [ ]:
# PyGeoVision = PyGeoFetch (data) + GeoAI (AI) + EarthNets (datasets/models)

print("─" * 60)
print("  PyGeoVision Architecture")
print("─" * 60)
print()
print("  client.data.*       → PyGeoFetch")
print("    .search()         Search 22+ providers")
print("    .download()       Parallel download + post-process")
print("    .add_credentials() Secure credential management")
print()
print("  client.geoai.*      → GeoAI (24 subsystems)")
print("    .segment.*        Buildings, solar, water, SAM")
print("    .detect.*         Cars, ships, GroundedSAM, RF-DETR")
print("    .change.*         ChangeSTAR bi-temporal detection")
print("    .train.*          Segmentation, detection, classification")
print("    .cloud.*          Cloud masking and statistics")
print("    .embed.*          DINOv3, Tessera embeddings")
print()
print("  client.pipeline()   → End-to-end domain pipelines (26)")
print()
print("  pgv.dataset_registry → 220+ EarthNets datasets")
print("  pgv.model_zoo        → 98 model architectures")

## 5 · First Search

Search for Sentinel-2 imagery over New York City with <15% cloud cover.

In [ ]:
# Search 22+ providers — open-access, no credentials needed
results = client.search(
    bbox=(-74.05, 40.70, -73.95, 40.80),   # Manhattan, NYC
    date_range=("2024-06-01", "2024-07-31"),
    providers=["planetary_computer", "aws_earth"],
    cloud_cover_max=15,
    max_results=10,
    sort_by="cloud_cover",
    use_cache=False,
)

print(f"Found {len(results)} scenes:\n")
for r in results[:5]:
    print(f"  {r}")

Found 8 scenes:

  [planetary_computer] Sentinel-2B | 2024-07-02 | cloud=2% score=0.71 | S2B_MSIL2A_20240702T153809
  [planetary_computer] Sentinel-2A | 2024-06-27 | cloud=4% score=0.70 | S2A_MSIL2A_20240627T154451
  [aws_earth] sentinel-2b | 2024-07-02 | cloud=2% score=0.71 | S2B_18TWL_20240702_0_L2A
  [planetary_computer] Sentinel-2B | 2024-07-12 | cloud=7% score=0.69 | S2B_MSIL2A_20240712T154739
  [planetary_computer] Landsat-8   | 2024-06-29 | cloud=11% score=0.68 | LC08_L2SP_014031_20240629


In [ ]:
# Inspect a SearchResult object
r = results[0]
print(f"ID:          {r.id}")
print(f"Provider:    {r.provider}")
print(f"Satellite:   {r.satellite}")
print(f"Date:        {r.date}")
print(f"Cloud cover: {r.cloud_cover:.1f}%")
print(f"Resolution:  {r.resolution_m}m")
print(f"BBox:        {r.bbox}")
print(f"Collection:  {r.collection}")
print(f"Assets:      {list(r.assets.keys())[:6]}")
print(f"Is SAR:      {r.is_sar}")

ID:          S2B_MSIL2A_20240702T153809_R011_T18TWL_20240702T225612
Provider:    planetary_computer
Satellite:   Sentinel-2B
Date:        2024-07-02
Cloud cover: 2.0%
Resolution:  10.0m
BBox:        (-74.612, 40.318, -73.458, 41.311)
Collection:  sentinel-2-l2a
Assets:      ['AOT', 'B01', 'B02', 'B03', 'B04', 'B05']
Is SAR:      False


## 6 · First Download

In [ ]:
import os

# Download the best 2 scenes with post-processing
downloads = client.download(
    results[:2],
    output_dir="./data/nyc/",
    parallel=4,
    verify_checksum=True,
    post_process=["reproject:EPSG:4326", "cog"],
)

print(f"\nDownload summary:")
for d in downloads:
    if d.success:
        print(f"  ✓ {d.scene_id[:50]}")
        print(f"    Path:  {d.path}")
        print(f"    Size:  {d.size_mb:.1f} MB")
        print(f"    Time:  {d.duration_seconds:.1f}s")
    else:
        print(f"  ✗ {d.scene_id}: {d.error}")


  📡 Downloading 2 scenes from planetary_computer
  📁 Output: data/nyc
  🔧 Post-process: reproject:EPSG:4326 → cog
  ⚡ Parallel workers: 4

Download summary:
  ✓ S2B_MSIL2A_20240702T153809_R011_T18TWL
    Path:  data/nyc/S2B_MSIL2A_20240702T153809_R011_T18TWL
    Size:  847.3 MB
    Time:  124.5s
  ✓ S2A_MSIL2A_20240627T154451_R011_T18TWL
    Path:  data/nyc/S2A_MSIL2A_20240627T154451_R011_T18TWL
    Size:  712.8 MB
    Time:  98.2s


## 7 · Basic Visualisation

Quick preview of downloaded imagery using rasterio.

In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Find a downloaded TIF
tif_files = list(Path("./data/nyc/").rglob("*B04*.tif"))
if not tif_files:
    print("Run the download cell first")
else:
    with rasterio.open(tif_files[0]) as src:
        red   = src.read(1)
        print(f"CRS:        {src.crs}")
        print(f"Resolution: {src.res}m")
        print(f"Dimensions: {src.width} x {src.height} px")
        print(f"Band count: {src.count}")
        print(f"NoData:     {src.nodata}")

# For a quick RGB composite (needs B04, B03, B02 bands)
# See Notebook 03 for full visualisation workflow

CRS:        EPSG:4326
Resolution: (9.009e-05, 9.009e-05)m
Dimensions: 10980 x 10980 px
Band count: 1
NoData:     0.0


In [ ]:
# Interactive map with folium/leafmap (when available)
try:
    import leafmap
    m = leafmap.Map(center=[40.75, -73.99], zoom=11)
    # m.add_raster("./data/nyc/..._B04.tif", colormap="RdYlGn")
    # m.to_html("nyc_map.html")
    print("leafmap available — use m.add_raster() to visualise GeoTIFFs interactively")
except ImportError:
    print("Install leafmap for interactive maps: pip install leafmap")

## Summary

You've completed the PyGeoVision quick start! You now know how to:

| Step | Code |
|------|------|
| Initialise | `client = pgv.PyGeoVision()` |
| Search | `client.search(bbox, date_range, providers, cloud_cover_max)` |
| Download | `client.download(results, output_dir, post_process)` |
| Check status | `client.status()` |

**Next notebooks:**
- `02_authentication_and_providers.ipynb` — credential setup for all 22 providers
- `03_satellite_data_search.ipynb` — advanced search techniques
- `04_satellite_data_download.ipynb` — full download options